In [ ]:
%load_ext autoreload

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

In [ ]:
%autoreload 2

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
import folium

# ------------------------------------------------------------------
# inputs
# ------------------------------------------------------------------
gdb = "/Volumes/x10pro/estuary/drivers_data/streamstats_polygons/steamstats12240.gdb/"
orig_path = Path("/Volumes/x10pro/estuary/drivers_data/geos/nearest_streamstats_points.csv")
huc_path = "/Volumes/x10pro/estuary/drivers_data/WBD_18_HU2_Shape/Shape/WBDHU12.shp"

# ------------------------------------------------------------------
# load original points
# ------------------------------------------------------------------
suffix = orig_path.suffix.lower()

if suffix == ".parquet":
    orig_df = pd.read_parquet(orig_path)
elif suffix == ".csv":
    orig_df = pd.read_csv(orig_path)
else:
    orig_df = gpd.read_file(orig_path)
    if "geometry" in orig_df.columns and (
        "pt_lon" not in orig_df.columns or "pt_lat" not in orig_df.columns
    ):
        orig_df = orig_df.copy()
        orig_df["pt_lon"] = orig_df.geometry.x
        orig_df["pt_lat"] = orig_df.geometry.y

required_cols = ["site_id", "pt_lon", "pt_lat"]
missing = [c for c in required_cols if c not in orig_df.columns]
if missing:
    raise ValueError(f"Original points file missing columns: {missing}")

orig_gdf = gpd.GeoDataFrame(
    orig_df.copy(),
    geometry=gpd.points_from_xy(orig_df["pt_lon"], orig_df["pt_lat"]),
    crs="EPSG:4326",
)

orig_gdf["site_id"] = orig_gdf["site_id"].astype(int)

# ------------------------------------------------------------------
# load StreamStats outputs
# ------------------------------------------------------------------
gdf_points = gpd.read_file(gdb, layer="GlobalWatershedPoint").to_crs(4326)
gdf_poly = gpd.read_file(gdb, layer="GlobalWatershed").to_crs(4326)

gdf_points = gdf_points.rename(columns={"Name": "site_id"})
gdf_poly = gdf_poly.rename(columns={"Name": "site_id"})

gdf_points["site_id"] = gdf_points["site_id"].astype(int)
gdf_poly["site_id"] = gdf_poly["site_id"].astype(int)

gdf_points = gdf_points.sort_values("site_id").drop_duplicates(subset="site_id")
gdf_poly = gdf_poly.sort_values("site_id").drop_duplicates(subset="site_id")

# ------------------------------------------------------------------
# load HUC12 polygons and subset to those intersecting original points
# ------------------------------------------------------------------
huc_polys = gpd.read_file(huc_path)[["huc12", "name", "geometry"]].to_crs(4326).iloc[:1]

joined_huc = gpd.sjoin(
    huc_polys,
    orig_gdf[["geometry"]],
    how="inner",
    predicate="intersects",
)

huc_polys_subset = huc_polys.loc[joined_huc.index.unique()].copy()

# ------------------------------------------------------------------
# join status
# ------------------------------------------------------------------
hit_ids = set(gdf_poly["site_id"])
orig_gdf["has_streamstats_polygon"] = orig_gdf["site_id"].isin(hit_ids)

plot_df = orig_gdf.merge(
    gdf_points[["site_id", "WarningMsg"]],
    on="site_id",
    how="left",
)

# ------------------------------------------------------------------
# map
# ------------------------------------------------------------------
center = [orig_gdf.geometry.y.mean(), orig_gdf.geometry.x.mean()]
m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")


# HUC12 polygons in blue
def huc_style(_):
    return {
        "color": "blue",
        "weight": 1,
        "fillColor": "blue",
        "fillOpacity": 0.12,
    }


if len(huc_polys_subset) > 0:
    folium.GeoJson(
        huc_polys_subset[["huc12", "name", "geometry"]].to_json(),
        name="HUC12 watersheds",
        style_function=huc_style,
        tooltip=folium.GeoJsonTooltip(
            fields=["huc12", "name"],
            aliases=["huc12", "name"],
            sticky=False,
        ),
    ).add_to(m)


# StreamStats polygons in green
def ss_style(_):
    return {
        "color": "green",
        "weight": 2,
        "fillColor": "green",
        "fillOpacity": 0.15,
    }


if len(gdf_poly) > 0:
    ss_fields = [c for c in ["site_id", "DRNAREA", "WarningMsg"] if c in gdf_poly.columns]
    ss_aliases = ss_fields.copy()

    folium.GeoJson(
        gdf_poly[ss_fields + ["geometry"]].to_json(),
        name="StreamStats polygons",
        style_function=ss_style,
        tooltip=folium.GeoJsonTooltip(
            fields=ss_fields,
            aliases=ss_aliases,
            sticky=False,
        ),
    ).add_to(m)

# Original points only
for row in plot_df.itertuples(index=False):
    color = "green" if row.has_streamstats_polygon else "red"
    warning = getattr(row, "WarningMsg", None)

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.95,
        tooltip=f"{row.site_id} | {'hit' if row.has_streamstats_polygon else 'miss'}",
        popup=(
            f"<b>site_id:</b> {row.site_id}<br>"
            f"<b>status:</b> {'hit' if row.has_streamstats_polygon else 'miss'}<br>"
            f"<b>orig lon:</b> {row.geometry.x:.6f}<br>"
            f"<b>orig lat:</b> {row.geometry.y:.6f}<br>"
            f"<b>warning:</b> {warning if pd.notna(warning) else ''}"
        ),
    ).add_to(m)

legend_html = """
<div style="
position: fixed;
bottom: 40px; left: 40px; width: 240px; height: 120px;
background-color: white; z-index:9999; font-size:14px;
border:2px solid grey; padding: 10px;">
<b>Legend</b><br>
<span style="color:green;">●</span> Original point, StreamStats hit<br>
<span style="color:red;">●</span> Original point, StreamStats miss<br>
<span style="color:green;">▉</span> StreamStats polygon<br>
<span style="color:blue;">▉</span> HUC12 watershed
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(m)
m

In [ ]:
from pathlib import Path

import folium
import pandas as pd

# read output
path = Path("/Volumes/x10pro/estuary/drivers_data/geos/nearest_streamstats_points.csv")
df = pd.read_csv(path)
# df["mouth_lon_360"] = (df["mouth_lon"] + 360) % 360
# df["pt_lon_360"] = (df["pt_lon"] + 360) % 360
# df = df.sort_values("dist_km", ascending=False).tail(10)

# folium wants longitudes in [-180, 180]
df = df.copy()
df["mouth_lon_plot"] = ((df["mouth_lon_360"] + 180) % 360) - 180
df["pt_lon_plot"] = ((df["pt_lon_360"] + 180) % 360) - 180

center_lat = df["mouth_lat"].mean()
center_lon = df["mouth_lon_plot"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=6,
    tiles="CartoDB positron",
)

for row in df.itertuples(index=False):
    popup = (
        f"<b>{row.site_id}</b><br>Candidate: {row.candidate_name}<br>Distance: {row.dist_km:.2f} km"
    )

    folium.PolyLine(
        locations=[
            [row.mouth_lat, row.mouth_lon_plot],
            [row.pt_lat, row.pt_lon_plot],
        ],
        weight=2,
        opacity=0.7,
        popup=popup,
    ).add_to(m)

    folium.CircleMarker(
        location=[row.mouth_lat, row.mouth_lon_plot],
        radius=5,
        color="blue",
        fill=True,
        fill_opacity=0.9,
        tooltip=f"{row.site_id}: selected candidate ({row.candidate_name})",
        popup=(
            f"<b>{row.site_id}</b><br>"
            f"Selected candidate: {row.candidate_name}<br>"
            f"Mouth lon: {row.mouth_lon_plot:.5f}<br>"
            f"Mouth lat: {row.mouth_lat:.5f}"
        ),
    ).add_to(m)

    folium.CircleMarker(
        location=[row.pt_lat, row.pt_lon_plot],
        radius=4,
        color="red",
        fill=True,
        fill_opacity=0.9,
        tooltip=f"{row.site_id}: nearest FES point",
        popup=(
            f"<b>{row.site_id}</b><br>"
            f"Nearest FES point<br>"
            f"Point lon: {row.pt_lon_plot:.5f}<br>"
            f"Point lat: {row.pt_lat:.5f}<br>"
            f"Distance: {row.dist_km:.2f} km"
        ),
    ).add_to(m)

legend_html = """
<div style="
position: fixed;
bottom: 40px; left: 40px; width: 190px; height: 95px;
background-color: white; z-index:9999; font-size:14px;
border:2px solid grey; padding: 10px;
">
<b>Legend</b><br>
<span style="color:blue;">●</span> Selected site candidate<br>
<span style="color:red;">●</span> Nearest FES point<br>
<span style="color:black;">━</span> Connection
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m

In [ ]:
usgs_gdf.head()

In [ ]:
from pathlib import Path

import folium
import geopandas as gpd
import pandas as pd

# ------------------------------------------------------------------
# inputs
# ------------------------------------------------------------------
gdb = "/Volumes/x10pro/estuary/drivers_data/streamstats_polygons/steamstats12240.gdb/"
orig_path = Path("/Volumes/x10pro/estuary/drivers_data/geos/nearest_streamstats_points.csv")
usgs_path = "/Volumes/x10pro/estuary/drivers_data/geos/nearest_usgs_gauges.geojson"

# ------------------------------------------------------------------
# load original outlet points
# ------------------------------------------------------------------
suffix = orig_path.suffix.lower()

if suffix == ".parquet":
    orig_df = pd.read_parquet(orig_path)
elif suffix == ".csv":
    orig_df = pd.read_csv(orig_path)
else:
    orig_df = gpd.read_file(orig_path)
    if "geometry" in orig_df.columns and (
        "pt_lon" not in orig_df.columns or "pt_lat" not in orig_df.columns
    ):
        orig_df = orig_df.copy()
        orig_df["pt_lon"] = orig_df.geometry.x
        orig_df["pt_lat"] = orig_df.geometry.y

required_cols = ["site_id", "pt_lon", "pt_lat"]
missing = [c for c in required_cols if c not in orig_df.columns]
if missing:
    raise ValueError(f"Original points file missing columns: {missing}")

orig_gdf = gpd.GeoDataFrame(
    orig_df.copy(),
    geometry=gpd.points_from_xy(orig_df["pt_lon"], orig_df["pt_lat"]),
    crs="EPSG:4326",
)
orig_gdf["site_id"] = orig_gdf["site_id"].astype(int)
orig_gdf = orig_gdf.sort_values("site_id").drop_duplicates(subset="site_id")

# ------------------------------------------------------------------
# load StreamStats polygons
# ------------------------------------------------------------------
gdf_poly = gpd.read_file(gdb, layer="GlobalWatershed").to_crs(4326)
gdf_poly = gdf_poly.rename(columns={"Name": "site_id"})
gdf_poly["site_id"] = gdf_poly["site_id"].astype(int)
gdf_poly = gdf_poly.sort_values("site_id").drop_duplicates(subset="site_id")

# ------------------------------------------------------------------
# load USGS gauges
# ------------------------------------------------------------------
usgs_gdf = gpd.read_file(usgs_path).to_crs(4326)

if "site_id" not in usgs_gdf.columns:
    raise ValueError("USGS file is missing site_id")
usgs_gdf["site_id"] = usgs_gdf["site_id"].astype(int)

# optional: if you want only one gauge per site, uncomment this
# usgs_gdf = usgs_gdf.sort_values(["site_id", "distance_to_outlet_m"]).drop_duplicates("site_id")
# usgs_gdf = usgs_gdf[usgs_gdf.distance_to_outlet_m < 25000].copy()

# ------------------------------------------------------------------
# join gauge -> original point for line drawing
# ------------------------------------------------------------------
line_df = usgs_gdf.merge(
    orig_gdf[["site_id", "geometry"]].rename(columns={"geometry": "outlet_geometry"}),
    on="site_id",
    how="left",
)

# ------------------------------------------------------------------
# map
# ------------------------------------------------------------------
center = [orig_gdf.geometry.y.mean(), orig_gdf.geometry.x.mean()]
m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")


# StreamStats polygons in green
def ss_style(_):
    return {
        "color": "green",
        "weight": 2,
        "fillColor": "green",
        "fillOpacity": 0.12,
    }


if len(gdf_poly) > 0:
    ss_fields = [c for c in ["site_id", "DRNAREA", "WarningMsg"] if c in gdf_poly.columns]
    ss_aliases = ss_fields.copy()

    folium.GeoJson(
        gdf_poly[ss_fields + ["geometry"]].to_json(),
        name="StreamStats polygons",
        style_function=ss_style,
        # tooltip=folium.GeoJsonTooltip(
        #     fields=ss_fields,
        #     aliases=ss_aliases,
        #     sticky=False,
        # ),
    ).add_to(m)

# lines: USGS gauge -> assigned outlet point
for row in line_df.itertuples(index=False):
    if row.outlet_geometry is None or row.geometry is None:
        continue

    folium.PolyLine(
        locations=[
            [row.geometry.y, row.geometry.x],  # gauge
            [row.outlet_geometry.y, row.outlet_geometry.x],  # outlet
        ],
        color="gray",
        weight=1,
        opacity=0.7,
        tooltip=f"site_id {row.site_id}",
    ).add_to(m)

# original outlet points
for row in orig_gdf.itertuples(index=False):
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.95,
        tooltip=f"Outlet {row.site_id}",
        popup=(
            f"<b>site_id:</b> {row.site_id}<br>"
            f"<b>outlet lon:</b> {row.geometry.x:.6f}<br>"
            f"<b>outlet lat:</b> {row.geometry.y:.6f}"
        ),
    ).add_to(m)

# USGS gauges
for row in usgs_gdf.itertuples(index=False):
    dist_txt = ""
    if hasattr(row, "distance_to_outlet_m") and pd.notna(row.distance_to_outlet_m):
        dist_txt = f"<b>distance to outlet (m):</b> {row.distance_to_outlet_m:.1f}<br>"

    types_txt = ""
    if hasattr(row, "data_types_found") and pd.notna(row.data_types_found):
        types_txt = f"<b>data types:</b> {row.data_types_found}<br>"

    gauge_color = "blue" if getattr(row, "has_iv", False) else "purple"
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="blue",
        fill=True,
        fill_color=gauge_color,
        fill_opacity=0.9,
        tooltip=f"USGS {row.usgs_site_no} | site_id {row.site_id} | dist {row.distance_to_outlet_m:1f} | {row.site_name}",
        popup=(
            f"<b>site_id:</b> {row.site_id}<br>"
            f"<b>USGS site:</b> {row.usgs_site_no}<br>"
            f"<b>name:</b> {getattr(row, 'site_name', '')}<br>"
            f"{types_txt}"
            f"{dist_txt}"
            f"<b>iv_first_time: {row.iv_first_time.date()}<br>"
        ),
    ).add_to(m)

legend_html = """
<div style="
position: fixed;
bottom: 40px; left: 40px; width: 250px; height: 120px;
background-color: white; z-index:9999; font-size:14px;
border:2px solid grey; padding: 10px;">
<b>Legend</b><br>
<span style="color:green;">▉</span> StreamStats polygon<br>
<span style="color:red;">●</span> Outlet point<br>
<span style="color:blue;">●</span> USGS gauge<br>
<span style="color:gray;">━</span> Gauge → assigned outlet
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(m)
m

In [ ]:
from pathlib import Path

import pandas as pd

path = Path("/Volumes/x10pro/estuary/drivers_data/usgs_data_iv.parquet")
df = pd.read_parquet(path)

print(df.shape)
df.head()

In [ ]:
df.dtypes

In [ ]:
df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce", utc=True)
df["value_num"] = pd.to_numeric(df["value"], errors="coerce")

print("rows:", len(df))
print("unique site_ids:", df["site_id"].nunique())
print("unique gauge_ids:", df["gauge_id"].nunique())
print("date_time nulls:", df["date_time"].isna().sum())
print("value_num nulls:", df["value_num"].isna().sum())

In [ ]:
SITE_GAUGES: dict[int, list[str]] = {
    96: ["11482500"],
    12103: ["11481200"],
    92: ["11469000"],
    84: ["11468000"],
    77: ["11467553", "11467510"],
    72: ["11467000", "11467200"],
    13057: ["11162570"],
    2138: ["11162500"],
    56: ["11161000"],
    57: ["11160000"],
    51: ["11159500"],
    50: ["11152500"],
    48: ["11143250"],
    33: ["11140585"],
    32: ["11136100"],
    31: ["11134000"],
    28: ["11120000", "11119940", "11120500", "11120520"],
    27: ["11119770", "11119750", "11119745"],
    22: ["11119500"],
    21: ["11118500"],
    20: ["11109000", "11113000", "11113500"],
    17: ["11046325", "11046360", "11046300"],
    16: ["11046100"],
    15: ["11046000"],
    14: ["11042000"],
    11: ["11023340"],
}

expected = sorted({g for gauges in SITE_GAUGES.values() for g in gauges})
found = sorted(df["gauge_id"].astype(str).unique())

missing = sorted(set(expected) - set(found))
extra = sorted(set(found) - set(expected))

print("expected gauges:", len(expected))
print("found gauges:", len(found))
print("missing:", missing)
print("extra:", extra)

In [ ]:
gauge_summary = (
    df.groupby(["site_id", "gauge_id", "data_type"], dropna=False)
    .agg(
        n_rows=("value", "size"),
        n_nonnull_values=("value_num", lambda s: s.notna().sum()),
        n_variables=("variable_code", "nunique"),
        first_time=("date_time", "min"),
        last_time=("date_time", "max"),
        site_name=("site_name", "first"),
    )
    .reset_index()
    .sort_values(["site_id", "gauge_id", "data_type"])
)

gauge_summary

In [ ]:
var_summary = (
    df.groupby(["gauge_id", "data_type", "variable_code", "variable_name", "unit"], dropna=False)
    .size()
    .reset_index(name="n_rows")
    .sort_values(["gauge_id", "data_type", "variable_code"])
)

var_summary

In [ ]:
bad_gauges = (
    df.groupby(["site_id", "gauge_id", "data_type"])
    .agg(
        n_rows=("value", "size"),
        n_numeric=("value_num", lambda s: s.notna().sum()),
    )
    .reset_index()
)

bad_gauges.loc[bad_gauges["n_numeric"] == 0].sort_values(["site_id", "gauge_id"])

In [ ]:
coverage = (
    df.groupby(["site_id", "gauge_id", "data_type"])
    .agg(
        first_time=("date_time", "min"),
        last_time=("date_time", "max"),
    )
    .reset_index()
)

coverage["covers_start"] = coverage["first_time"] <= pd.Timestamp("2018-01-01", tz="UTC")
coverage["covers_end"] = coverage["last_time"] >= pd.Timestamp("2024-12-31", tz="UTC")

coverage.sort_values(["site_id", "gauge_id", "data_type"])

In [ ]:
pivot = (
    gauge_summary.pivot_table(
        index=["site_id", "gauge_id", "site_name"],
        columns="data_type",
        values="n_rows",
        aggfunc="first",
    )
    .reset_index()
    .sort_values(["site_id", "gauge_id"])
)

pivot

In [ ]:
df.loc[df["gauge_id"].astype(str) == "11482500"].sort_values("date_time").head(20)

In [ ]:
g = "11482500"
plot_df = df[
    (df["gauge_id"].astype(str) == g)
    & (df["data_type"] == "iv")
    & (df["variable_code"].astype(str) == "00060")
].copy()

plot_df = plot_df.sort_values("date_time")

plt.figure(figsize=(10, 4))
plt.plot(plot_df["date_time"], plot_df["value_num"])
plt.title(f"{g} IV discharge")
plt.xlabel("time")
plt.ylabel("value")
plt.show()